# Deploying fastapifromfrictionless with Podman

This notebook walks through the full deployment workflow: from writing your first schema files
to a running API backed by a PostGIS database, all managed by Podman Compose.

By the end you will have three containers running on your machine:

| Container | What it does | URL |
|-----------|-------------|-----|
| `postgres` | PostGIS database that stores your data | `localhost:5432` |
| `api` | Generated FastAPI application with CRUD endpoints | `http://localhost:8000` |
| `pgadmin` | Web-based database browser | `http://localhost:8080` |

> **Prerequisites** — install Podman and podman-compose before starting:
> ```bash
> sudo apt -y install podman podman-compose   # Linux
> ```
> For Windows, follow the [Podman for Windows tutorial](https://github.com/containers/podman/blob/main/docs/tutorials/podman-for-windows.md).

## Step 1 — Create a deployment folder

The `podman/` folder in this repo contains everything needed to run the stack.
Copy it to a new directory — this becomes your deployment home.

```bash
cp -r path/to/fastapi-from-frictionless/podman/ my-deployment/
cd my-deployment/
```

Your folder should look like this:

```
my-deployment/
├── compose.yaml        ← orchestrates the three containers
├── Dockerfile          ← builds the API image
├── entrypoint.sh       ← generates the app from your schemas at startup
├── .env.example        ← template for passwords and settings
└── schemas/            ← your *.schema.yaml files go here
```

## Step 2 — Define your data with schemas

A Frictionless schema is a short YAML file that describes one table — its fields, types,
required constraints, and relationships to other tables. You need one file per resource.

The cell below writes two example schemas to the `schemas/` folder:
**location** (where sensors are installed) and **sensor** (linked to a location).
Replace these with your own data once you have the workflow running.

In [ ]:
import pathlib

schemas = pathlib.Path("schemas")
schemas.mkdir(exist_ok=True)

(schemas / "location.schema.yaml").write_text("""\
fields:
  - name: name
    type: string
    constraints:
      required: true
  - name: address
    type: string
    constraints:
      required: true
primaryKey:
  - name
""")

(schemas / "sensor.schema.yaml").write_text("""\
fields:
  - name: id
    type: integer
    constraints:
      required: true
  - name: label
    type: string
    constraints:
      required: true
  - name: location_name
    type: string
    constraints:
      required: true
  - name: status
    type: string
primaryKey:
  - id
foreignKeys:
  - fields: [location_name]
    reference:
      resource: location
      fields: [name]
""")

for f in sorted(schemas.iterdir()):
    print(f.name)
    print(f.read_text())

The `sensor` schema declares a **foreign key** pointing at `location`. The tool uses this to:
- Enforce referential integrity at the database level
- Generate a `SensorPublicWithAll` response model that includes the nested location
- Add a `GET /sensor/query` endpoint for filtered, joined queries

A schema with no foreign keys (like `location`) gets the simpler set of endpoints without the query route.

## Step 3 — Configure environment variables

Copy `.env.example` to `.env` and fill in your passwords.

```bash
cp .env.example .env
```

Edit `.env` — every value marked `CHANGE_ME` must be replaced:

In [ ]:
env_example = """
# PostgreSQL credentials
POSTGRES_USER=postgres
POSTGRES_PASSWORD=a_long_random_string_here
POSTGRES_DB=mydb

# pgAdmin login
PGADMIN_DEFAULT_EMAIL=you@example.com
PGADMIN_DEFAULT_PASSWORD=another_long_random_string

# API settings
API_KEY=                        # leave empty to disable authentication
ALLOWED_ORIGINS=*               # comma-separated origins, or * for open dev access
API_URL=http://localhost:8000   # public base URL (used by Excel export)
"""
print(env_example)

> **Security note**: `.env` is listed in `.gitignore` — never commit it.
>
> `API_KEY` is optional. Leave it blank during development. When set, every API request
> must include the header `X-API-Key: <your value>` or it will be rejected with HTTP 403.

## Step 4 — Start the stack

One command pulls the images, builds the API container, and starts all three services.

```bash
podman-compose up -d
```

The first run takes a couple of minutes to download the PostGIS and pgAdmin images.
Subsequent starts are fast.

**What happens at startup:**
1. `postgres` starts and creates the database
2. `api` waits for postgres to pass its health check, then:
   - Reads your `schemas/` folder
   - Generates `models.py`, `app.py`, and `database.py`
   - Runs `uvicorn` on port 8000
3. `pgadmin` starts independently on port 8080

Check that all three are running:

```bash
podman-compose ps
```

Watch the API logs to confirm startup succeeded:

```bash
podman-compose logs api
```

You should see a line like `Application startup complete.`

## Step 5 — Explore the API

FastAPI generates interactive documentation automatically. Open your browser to:

> **http://localhost:8000/docs**

You will see every endpoint for every schema — try creating a record directly from the docs UI.

The cells below show the same operations using Python's `requests` library.
They will only work once the stack is running.

In [ ]:
import requests

BASE = "http://localhost:8000"

# Verify the API is reachable
try:
    r = requests.get(f"{BASE}/location/all", timeout=3)
    print(f"API is up — status {r.status_code}")
except requests.exceptions.ConnectionError:
    print("API is not reachable yet. Start the stack with: podman-compose up -d")

In [ ]:
# Create locations
locations = [
    {"name": "City Hall",   "address": "90 W Broad St, Columbus OH"},
    {"name": "Short North", "address": "1200 N High St, Columbus OH"},
]

for loc in locations:
    r = requests.post(f"{BASE}/location", json=loc)
    print(r.status_code, r.json())

In [ ]:
# Create sensors (location_name must match an existing location)
sensors = [
    {"id": 1, "label": "AQ-01", "location_name": "City Hall",   "status": "active"},
    {"id": 2, "label": "AQ-02", "location_name": "Short North", "status": "active"},
    {"id": 3, "label": "WX-01", "location_name": "City Hall",   "status": "maintenance"},
]

for s in sensors:
    r = requests.post(f"{BASE}/sensor", json=s)
    print(r.status_code, r.json())

In [ ]:
# Read data — each resource gets /all, /recent, and /{pk} endpoints
print("All sensors:")
for item in requests.get(f"{BASE}/sensor/all").json():
    print(" ", item)

print("\nMost recently added (limit=2):")
for item in requests.get(f"{BASE}/sensor/recent", params={"limit": 2}).json():
    print(" ", item)

print("\nUpdate sensor 3 status:")
r = requests.patch(f"{BASE}/sensor/3", json={"status": "active"})
print(" ", r.json())

## Step 6 — Browse the database with pgAdmin

pgAdmin is a web-based UI for exploring and querying the Postgres database directly.

1. Open **http://localhost:8080** in your browser
2. Log in with the email and password you set in `.env` (`PGADMIN_DEFAULT_EMAIL` / `PGADMIN_DEFAULT_PASSWORD`)
3. Click **Add New Server** (or right-click Servers → Register → Server)
4. In the **General** tab, give it a name (e.g. `app-db`)
5. In the **Connection** tab:
   - **Host**: `10.91.0.5`  *(the postgres container's static IP)*
   - **Port**: `5432`
   - **Username**: value of `POSTGRES_USER` from `.env`
   - **Password**: value of `POSTGRES_PASSWORD` from `.env` — check "Save password"
6. Click **Save**

Your tables (`location`, `sensor`, etc.) appear under **Servers → app-db → Databases → mydb → Schemas → public → Tables**.

pgAdmin saves its state to `./pgadmin/` on your host machine, so the connection survives container restarts.

## Step 7 — Excel import and export

The API has built-in endpoints for moving data to and from Excel — useful when collaborators
prefer spreadsheets over API calls.

### Export all data to a workbook

```
GET /excel/export
```

Returns an `.xlsx` file with one sheet per resource, columns ordered to match the schema.

In [ ]:
import pandas as pd

r = requests.get(f"{BASE}/excel/export")
if r.ok:
    with open("export.xlsx", "wb") as f:
        f.write(r.content)
    print(f"Saved {len(r.content):,} bytes to export.xlsx\n")
    for sheet, df in pd.read_excel("export.xlsx", sheet_name=None).items():
        print(f"{sheet} ({len(df)} rows):")
        print(df.to_string(index=False), "\n")
else:
    print("Start the stack first: podman-compose up -d")

### Import data from a workbook

```
POST /excel/import
```

Upload a filled-in workbook and the API syncs it to the database — creating new rows and
patching changed ones.

```python
with open("my_data.xlsx", "rb") as f:
    r = requests.post(f"{BASE}/excel/import", files={"file": f})
print(r.json())   # {"status": "imported", "filename": "my_data.xlsx"}
```

The workbook must have one sheet per resource, with column names matching the schema field names.
Use the export as a starting template.

## Step 8 — Update your schemas

When you add or change schemas, the API image needs to be rebuilt — the code generation
runs at container startup, not at image build time, so:

```bash
# Stop the api container, regenerate, and restart
podman-compose restart api
```

If you change which schemas exist (add or remove a file), restart is enough because the
entrypoint re-runs `fastapifromfrictionless generate` every time the container starts.

If you update `fastapifromfrictionless` itself (e.g. after a new release):

```bash
podman-compose build api   # rebuilds the image with the latest package
podman-compose up -d api   # starts the new image
```

## Step 9 — Stop and clean up

```bash
# Stop all containers (data is preserved in ./postgres/ and ./pgadmin/)
podman-compose down

# Start again later
podman-compose up -d
```

To wipe everything and start fresh (this **deletes all data**):

```bash
podman-compose down
rm -rf ./postgres/ ./pgadmin/
```

---

## Quick reference

| Task | Command |
|------|---------|
| Start the stack | `podman-compose up -d` |
| Stop the stack | `podman-compose down` |
| View logs | `podman-compose logs api` |
| Restart API only | `podman-compose restart api` |
| Rebuild API image | `podman-compose build api` |
| Open API docs | http://localhost:8000/docs |
| Open pgAdmin | http://localhost:8080 |